# Non-Instruction Fine-Tuning Notebook
## Stage 1: Domain Adaptation

This notebook performs non-instruction fine-tuning on raw domain text to adapt the base model to course-specific language and terminology.

## Step 1: Install Required Libraries

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes accelerate unsloth[colab-new] -U

## Step 2: Check GPU and Import Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import json
import os
from pathlib import Path

print("=" * 60)
print("STAGE 1: NON-INSTRUCTION FINE-TUNING")
print("=" * 60)

print(f\"GPU Available: {torch.cuda.is_available()}\")
if torch.cuda.is_available():
    print(f\"GPU Name: {torch.cuda.get_device_name(0)}\")
    print(f\"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB\")
else:
    print("WARNING: No GPU detected. Training will be slow.")

## Step 3: Load Raw Domain Text

In [ ]:
print("\n[STEP 2] Loading raw domain text...")
data_path = 'course-doubt-assistant/data/non_instruction_data.txt'

with open(data_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split('\\n\\n') if p.strip()]
print(f\"✓ Loaded {len(paragraphs)} paragraphs\")
print(f\"✓ Total characters: {len(raw_text):,}\")
print(f\"\\nFirst paragraph:\\n{paragraphs[0][:200]}...\\n")

## Step 4: Create Dataset

In [ ]:
print("[STEP 3] Creating dataset...")
dataset_dict = {'text': paragraphs}
dataset = Dataset.from_dict(dataset_dict)
print(f\"✓ Dataset size: {len(dataset)}\")
print(f\"✓ Example: {dataset[0]['text'][:150]}...\\n")

## Step 5: Load Model and Tokenizer

In [ ]:
print("[STEP 4] Loading model and tokenizer...")
MODEL_NAME = 'unsloth/tinyllama-bnb-4bit'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)

print(f\"✓ Model loaded: {MODEL_NAME}\")
print(f\"✓ Model dtype: {model.dtype}\")
print(f\"✓ Model device: {next(model.parameters()).device}\\n")

## Step 6: Configure LoRA

In [ ]:
print("[STEP 5] Configuring LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
    modules_to_save=['lm_head']
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f\"✓ Trainable params: {trainable_params:,}\")
print(f\"✓ Total params: {total_params:,}\")
print(f\"✓ Trainable %: {100 * trainable_params / total_params:.2f}%\\n")

## Step 7: Tokenize Dataset

In [ ]:
print("[STEP 6] Tokenizing dataset...")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print(f\"✓ Tokenized dataset size: {len(tokenized_dataset)}\")
print(f\"✓ Keys: {tokenized_dataset.column_names}\")
print(f\"✓ Input IDs length: {len(tokenized_dataset[0]['input_ids'])}\\n")

## Step 8: Configure Training Arguments

In [ ]:
print("[STEP 7] Configuring training arguments...")
os.makedirs('./outputs/non_instruction_ft', exist_ok=True)

training_args = TrainingArguments(
    output_dir='./outputs/non_instruction_ft',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    optim='adamw_8bit',
    seed=42,
    report_to=[]
)

print("✓ Training arguments configured\\n")

## Step 9: Train Model

In [ ]:
print("[STEP 8] Starting training...")
print("-" * 60)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

train_result = trainer.train()
print("-" * 60)
print(f\"✓ Training loss: {train_result.training_loss:.4f}\\n")

## Step 10: Save Model and Adapter

In [ ]:
print("[STEP 9] Saving model and adapter...")
os.makedirs('./models/non_instruction_adapter', exist_ok=True)

adapter_path = './models/non_instruction_adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f\"✓ Adapter saved to {adapter_path}\")
print(f\"✓ Files: {os.listdir(adapter_path)}\\n")

## Step 11: Test Model After Non-Instruction FT

In [ ]:
print("[STEP 10] Testing non-instruction fine-tuned model...")
print("-" * 60)

def generate_response(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompts = [
    'Machine learning is',
    'Gradient descent',
    'Neural networks use'
]

for prompt in test_prompts:
    print(f\"\\nPrompt: {prompt}\")
    response = generate_response(prompt)
    print(f\"Response: {response}\")

print("\\n" + "=" * 60)
print("✓ STAGE 1 COMPLETE!")
print("=" * 60)